In [6]:
import pandas as pd
import altair as alt

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

df = pd.read_csv(url, header=None)

In [7]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [9]:
df = pd.read_csv(
    url,
    names=[
        "datetime",
        "city",
        "state",
        "country",
        "shape",
        "duration_seconds",
        "duration_text",
        "comments",
        "date_posted",
        "latitude",
        "longitude"
    ]
)

In [10]:
df.columns

Index(['datetime', 'city', 'state', 'country', 'shape', 'duration_seconds',
       'duration_text', 'comments', 'date_posted', 'latitude', 'longitude'],
      dtype='object')

In [11]:
df['shape'].value_counts().head(10)

shape
light        16565
triangle      7865
circle        7608
fireball      6208
other         5649
unknown       5584
sphere        5387
disk          5213
oval          3733
formation     2457
Name: count, dtype: int64

In [12]:
df['state'].value_counts().head(10)

state
ca    9655
wa    4268
fl    4200
tx    3677
ny    3219
az    2689
il    2645
pa    2582
oh    2425
mi    2071
Name: count, dtype: int64

In [13]:
df.head()

,datetime,city,state,country,shape,duration_seconds,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [14]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["year"] = df["datetime"].dt.year

df[["datetime","year"]].head()

,datetime,year
0,1949-10-10 20:30:00,1949
1,1949-10-10 21:00:00,1949
2,1955-10-10 17:00:00,1955
3,1956-10-10 21:00:00,1956
4,1960-10-10 20:00:00,1960


In [22]:
state_counts = (
    df.groupby(['state', 'shape'])
      .size()
      .reset_index(name='count')
)

shape_dropdown = alt.binding_select(
    options=sorted(state_counts["shape"].dropna().unique()),
    name="UFO Shape: "
)

shape_select = alt.selection_point(
    fields=["shape"],
    bind=shape_dropdown
)

plot1 = (
    alt.Chart(state_counts)
    .mark_bar()
    .encode(
        y=alt.Y(
            "state:N",
            sort="-x",
            title="State"
        ),
        x=alt.X(
            "count:Q",
            title="Number of Sightings"
        ),
        color=alt.Color(
            "count:Q",
            scale=alt.Scale(scheme="blues"),
            title="Sightings"
        )
    )
    .transform_filter(shape_select)
    .add_params(shape_select)
    .properties(
        title="UFO Sightings by State and Shape",
        width=700,
        height=500
    )
)

plot1

alt.Chart(...)

In [23]:
year_counts = (
    df.groupby('year')
      .size()
      .reset_index(name='count')
)

plot2 = (
    alt.Chart(year_counts)
    .mark_line(
        color="darkorange",
        strokeWidth=3
    )
    .encode(
        x=alt.X(
            "year:Q",
            title="Year"
        ),
        y=alt.Y(
            "count:Q",
            title="Number of Sightings"
        )
    )
    .properties(
        title="UFO Sightings Over Time",
        width=700,
        height=400
    )
)

plot2

alt.Chart(...)

In [24]:
plot1.save("ufo_shape_state.html")
plot2.save("ufo_year_trend.html")